# LỘ TRÌNH HỌC MACHINE LEARNING: BASIC DATA PREPROCESSING

Chào mừng bạn đến với phần học **Tiền xử lý dữ liệu (Data Preprocessing)**! 
Trong Machine Learning, có một câu nói nổi tiếng: **"Garbage In, Garbage Out"** (Dữ liệu rác đầu vào sẽ cho ra mô hình rác đầu ra). Khoảng 70% - 80% thời gian của một Kỹ sư Machine Learning / Data Scientist là dành cho việc làm sạch và tiền xử lý dữ liệu.

---

## TỔNG QUAN NỘI DUNG BÀI HỌC
1. **Tổng quan về Dữ liệu & Tiền xử lý dữ liệu (Introduction & Overview)**
2. **Xử lý dữ liệu thiếu (Handling Missing Data & Imputation)**
3. **Mã hóa dữ liệu phân loại (Categorical Data Encoding: Label & One-Hot Encoding)**
4. **Chuẩn hóa đặc trưng (Feature Scaling: Standardization vs Normalization)**
5. **Phân chia tập dữ liệu (Train - Test Split & Tránh Data Leakage)**

## 1. TỔNG QUAN VỀ DỮ LIỆU & TIỀN XỬ LÝ DỮ LIỆU

### 1.1 Phân loại dữ liệu trong Machine Learning
- **Dữ liệu dạng số (Numerical Data):**
  - **Continuous (Liên tục):** Chiều cao, cân nặng, giá nhà, nhiệt độ (ví dụ: 1m72, 65.5kg).
  - **Discrete (Rời rạc):** Số lượng con cái, số phòng ngủ (ví dụ: 1, 2, 3).
- **Dữ liệu phân loại (Categorical Data):**
  - **Nominal (Không thứ tự):** Màu sắc (Đỏ, Xanh, Vàng), Quốc tịch (Việt Nam, Mỹ, Nhật).
  - **Ordinal (Có thứ tự):** Trình độ học vấn (Tiểu học < Trung học < Đại học), Đánh giá (Tệ < Bình thường < Tốt).

### 1.2 Tại sao phải tiền xử lý dữ liệu?
Dữ liệu thực tế thường gặp các vấn đề:
- **Incomplete (Thiếu sót):** Thiếu giá trị ở một số cột/hàng (Missing values).
- **Inconsistent (Không nhất quán):** Tuổi là "25" nhưng năm sinh là "2010", định dạng ngày tháng lệch nhau.
- **Noisy (Nhiễu/Outlier):** Lỗi nhập liệu (ví dụ: Tuổi = -5 hoặc Chiều cao = 1700cm).
- **Non-numeric (Chữ/Chuỗi):** Hầu hết thuật toán ML chỉ tính toán được trên con số.

In [13]:
# Import các thư viện cơ bản
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tạo một Dataset mẫu có chứa dữ liệu lỗi, thiếu và dạng chữ để thực hành
raw_data = {
    'Age': [25, 30, np.nan, 45, 22, 38, np.nan, 50],
    'Salary': [50000, 64000, 58000, np.nan, 42000, 72000, 61000, 90000],
    'Education': ['Bachelor', 'Master', 'PhD', 'Bachelor', 'High School', 'Master', np.nan, 'Master'],
    'City': ['Hanoi', 'HCM', 'Hanoi', 'Da Nang', 'HCM', 'Da Nang', 'Hanoi', 'HCM'],
    'Purchased': ['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(raw_data)
print("--- DỮ LIỆU THÔ BAN ĐẦU ---")
df

--- DỮ LIỆU THÔ BAN ĐẦU ---


,Age,Salary,Education,City,Purchased
0,25.0,50000.0,Bachelor,Hanoi,No
1,30.0,64000.0,Master,HCM,Yes
2,NaN,58000.0,PhD,Hanoi,No
3,45.0,NaN,Bachelor,Da Nang,Yes
4,22.0,42000.0,High School,HCM,No
5,38.0,72000.0,Master,Da Nang,Yes
6,NaN,61000.0,NaN,Hanoi,No
7,50.0,90000.0,Master,HCM,Yes


## 2. XỬ LÝ DỮ LIỆU THIẾU (HANDLING MISSING DATA & IMPUTATION)

### Khái niệm:
Dữ liệu khuyết (Missing Data) xảy ra khi một trường thông tin không có giá trị (thường hiển thị là `NaN` hoặc `None`).

### Các phương pháp xử lý:
1. **Xóa (Dropping):**
   - Xóa dòng/cột bị thiếu (`df.dropna()`). Chỉ nên dùng khi lượng dữ liệu bị thiếu cực nhỏ (< 5%).
2. **Điền giá trị cơ bản (Simple Imputation):**
   - Cột số (Numerical): Điền bằng Trung bình (**Mean**), Trung vị (**Median**).
   - Cột phân loại (Categorical): Điền bằng Giá trị xuất hiện nhiều nhất (**Mode**).
3. **Điền giá trị nâng cao (Advanced Imputation):**
   - **KNNImputer**: Điền dựa trên các hàng lân cận có đặc trưng tương đồng nhất.
   - **IterativeImputer**: Dự đoán giá trị thiếu bằng mô hình hồi quy.

In [14]:
# 2.1 Kiểm tra số lượng giá trị thiếu trong mỗi cột
print("Số giá trị missing trong mỗi cột:")
print(df.isnull().sum())

# 2.2 Thực hành SimpleImputation bằng scikit-learn
from sklearn.impute import SimpleImputer

df_imputed = df.copy()

# Điền cột số (Age, Salary) bằng giá trị Mean (Trung bình)
num_imputer = SimpleImputer(strategy='mean')
df_imputed[['Age', 'Salary']] = num_imputer.fit_transform(df_imputed[['Age', 'Salary']])

# Điền cột phân loại (Education) bằng giá trị most_frequent (Mode/Xuất hiện nhiều nhất)
cat_imputer = SimpleImputer(strategy='most_frequent')
df_imputed[['Education']] = cat_imputer.fit_transform(df_imputed[['Education']])

print("\n--- DỮ LIỆU SAU KHI ĐIỀN MISSING VALUES (SIMPLE IMPUTATION) ---")
df_imputed

Số giá trị missing trong mỗi cột:
Age          2
Salary       1
Education    1
City         0
Purchased    0
dtype: int64

--- DỮ LIỆU SAU KHI ĐIỀN MISSING VALUES (SIMPLE IMPUTATION) ---


,Age,Salary,Education,City,Purchased
0,25.0,50000.000000,Bachelor,Hanoi,No
1,30.0,64000.000000,Master,HCM,Yes
2,35.0,58000.000000,PhD,Hanoi,No
3,45.0,62428.571429,Bachelor,Da Nang,Yes
4,22.0,42000.000000,High School,HCM,No
5,38.0,72000.000000,Master,Da Nang,Yes
6,35.0,61000.000000,Master,Hanoi,No
7,50.0,90000.000000,Master,HCM,Yes


In [15]:
# 2.3 Thực hành Advanced Imputation: KNNImputer (Điền dữ liệu bằng KNN)
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
age_salary_knn = knn_imputer.fit_transform(df[['Age', 'Salary']])
print("Age & Salary sau khi điền bằng KNNImputer:")
print(age_salary_knn)

Age & Salary sau khi điền bằng KNNImputer:
[[2.50e+01 5.00e+04]
 [3.00e+01 6.40e+04]
 [2.75e+01 5.80e+04]
 [4.50e+01 8.10e+04]
 [2.20e+01 4.20e+04]
 [3.80e+01 7.20e+04]
 [2.75e+01 6.10e+04]
 [5.00e+01 9.00e+04]]


## 3. MÃ HÓA DỮ LIỆU PHÂN LOẠI (CATEGORICAL DATA ENCODING)

Hầu hết mô hình Machine Learning chỉ nhận đầu vào là **con số**, không nhận được chuỗi chữ như `'Hanoi'`, `'Master'`. Do đó ta phải biến đổi (encode) chúng.

### 3.1 Label Encoding / Ordinal Encoding (Dành cho biến có thứ tự)
- Dùng khi dữ liệu có quan hệ hơn kém/thứ tự (ví dụ: High School = 0, Bachelor = 1, Master = 2, PhD = 3).
- **Lưu ý:** Không nên dùng Label Encoding cho dữ liệu **không thứ tự** (như tên thành phố), vì mô hình sẽ lầm tưởng Hanoi (0) < HCM (1) < Da Nang (2).

### 3.2 One-Hot Encoding (Dành cho biến KHÔNG có thứ tự)
- Tạo ra các cột mới kiểu Binary (0 hoặc 1) cho từng giá trị unique.
- Ví dụ cột City: `[City_Hanoi, City_HCM, City_Da Nang]`.
- **Tránh Dummy Variable Trap (Bẫy biến giả):** Khi cần thiết có thể loại bỏ 1 cột (`drop='first'`) vì nếu 2 cột bằng 0 thì chắc chắn cột thứ 3 bằng 1.

In [16]:
# 3.1 Áp dụng Ordinal Encoding cho cột Education (có thứ tự)
from sklearn.preprocessing import OrdinalEncoder

education_order = [['High School', 'Bachelor', 'Master', 'PhD']]
ordinal_enc = OrdinalEncoder(categories=education_order)

df_imputed['Education_Encoded'] = ordinal_enc.fit_transform(df_imputed[['Education']])

# 3.2 Áp dụng One-Hot Encoding cho cột City (không thứ tự)
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first') # drop='first' để tránh multicollinearity
city_ohe = ohe.fit_transform(df_imputed[['City']])
city_ohe_df = pd.DataFrame(city_ohe, columns=ohe.get_feature_names_out(['City']))

# Ghép các cột One-Hot vào DataFrame chính
df_encoded = pd.concat([df_imputed, city_ohe_df], axis=1)

# Mã hóa Label cho Target Column (Purchased: No=0, Yes=1)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_encoded['Purchased_Target'] = le.fit_transform(df_encoded['Purchased'])

print("--- DỮ LIỆU SAU KHI ENCODE CATEGORICAL FEATURES ---")
df_encoded[['Age', 'Salary', 'Education_Encoded', 'City_Hanoi', 'City_HCM', 'Purchased_Target']]

--- DỮ LIỆU SAU KHI ENCODE CATEGORICAL FEATURES ---


,Age,Salary,Education_Encoded,City_Hanoi,City_HCM,Purchased_Target
0,25.0,50000.000000,1.0,1.0,0.0,0
1,30.0,64000.000000,2.0,0.0,1.0,1
2,35.0,58000.000000,3.0,1.0,0.0,0
3,45.0,62428.571429,1.0,0.0,0.0,1
4,22.0,42000.000000,0.0,0.0,1.0,0
5,38.0,72000.000000,2.0,0.0,0.0,1
6,35.0,61000.000000,2.0,1.0,0.0,0
7,50.0,90000.000000,2.0,0.0,1.0,1


## 4. CHUẨN HÓA ĐẶC TRƯNG (FEATURE SCALING)

### Tại sao cần Feature Scaling?
Hãy nhìn vào 2 thuộc tính: `Age` (22 - 50) và `Salary` (42,000 - 90,000).
- Vì `Salary` có giá trị lớn hơn `Age` hàng nghìn lần, các thuật toán dựa trên khoảng cách (KNN, SVM, K-Means) hoặc Gradient Descent (Linear/Logistic Regression, Neural Networks) sẽ bị `Salary` chi phối hoàn toàn!
- Chuẩn hóa giúp đưa tất cả các đặc trưng về cùng một quy mô (scale).

### Các phương pháp chính:
1. **Min-Max Scaling (Normalization):**
   $$X_{new} = \frac{X - X_{min}}{X_{max} - X_{min}}$$
   - Đưa giá trị về khoảng $[0, 1]$. Độc hại với Outlier (nếu có outlier quá lớn sẽ kéo nén các dữ liệu còn lại).

2. **Standardization (Standard Scaling / Z-score):**
   $$X_{new} = \frac{X - \mu}{\sigma}$$
   - Đưa dữ liệu về dạng có Trung bình $\mu = 0$ và Độ lệch chuẩn $\sigma = 1$.
   - Phù hợp nhất cho đa số thuật toán Machine Learning (SVM, Logistic Regression, Neural Networks).

3. **Robust Scaling:**
   - Sử dụng Trung vị (Median) và Khoảng tứ phân vị (IQR).
   - Rất tốt khi dữ liệu chứa nhiều Outliers (giá trị ngoại lệ).

In [17]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

X_sample = df_encoded[['Age', 'Salary']]

scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()
scaler_robust = RobustScaler()

X_std = scaler_std.fit_transform(X_sample)
X_minmax = scaler_minmax.fit_transform(X_sample)
X_robust = scaler_robust.fit_transform(X_sample)

scale_demo = pd.DataFrame({
    'Age_Original': X_sample['Age'],
    'Age_Standardized': X_std[:, 0],
    'Age_MinMax': X_minmax[:, 0],
    'Age_Robust': X_robust[:, 0],
    'Salary_Original': X_sample['Salary'],
    'Salary_Standardized': X_std[:, 1],
    'Salary_MinMax': X_minmax[:, 1],
    'Salary_Robust': X_robust[:, 1]
})

print("--- SO SÁNH CÁC KỸ THUẬT SCALING ---")
scale_demo

--- SO SÁNH CÁC KỸ THUẬT SCALING ---


,Age_Original,Age_Standardized,Age_MinMax,Age_Robust,Salary_Original,Salary_Standardized,Salary_MinMax,Salary_Robust
0,25.0,-1.128665,0.107143,-0.909091,50000.000000,-0.923900,0.166667,-1.171429
1,30.0,-0.564333,0.285714,-0.454545,64000.000000,0.116815,0.458333,0.228571
2,35.0,0.000000,0.464286,0.000000,58000.000000,-0.329206,0.333333,-0.371429
3,45.0,1.128665,0.821429,0.909091,62428.571429,0.000000,0.425595,0.071429
4,22.0,-1.467265,0.000000,-1.181818,42000.000000,-1.518594,0.000000,-1.971429
5,38.0,0.338600,0.571429,0.272727,72000.000000,0.711509,0.625000,1.028571
6,35.0,0.000000,0.464286,0.000000,61000.000000,-0.106195,0.395833,-0.071429
7,50.0,1.692998,1.000000,1.363636,90000.000000,2.049570,1.000000,2.828571


## 5. PHÂN CHIA TẬP DỮ LIỆU (TRAIN-TEST SPLIT & DATA LEAKAGE)

### 5.1 Khái niệm:
- **Train Set (Tập huấn luyện):** Chiếm ~70-80% dữ liệu, dùng để mô hình học pattern.
- **Test Set (Tập kiểm tra):** Chiếm ~20-30% dữ liệu, đóng vai trò như "đề thi thật" để kiểm tra khả năng tổng quát hóa (generalization) của mô hình.

### 5.2 Quy tắc VÀNG tránh Data Leakage (Rò rỉ dữ liệu):
 **SAI LẦM PHỔ BIẾN:** Áp dụng `StandardScaler` hoặc `Imputer` trên TOÀN BỘ dataset trước khi chia Train-Test.
-> Điều này khiến thông tin của Test set (như Mean, Standard Deviation) rò rỉ vào Train set!

 **QUY TRÌNH CHUẨN:**
1. Chia dataset thành `X_train`, `X_test`, `y_train`, `y_test` trước.
2. Chỉ gọi `fit_transform()` trên `X_train`.
3. Chỉ gọi `transform()` trên `X_test` (dùng tham số đã học được từ X_train).

**Tập Test đóng vai trò giả lập "Khách hàng tương lai"**
Trong thực tế sản phẩm, khách hàng nộp hồ sơ tới từng người một. Bạn không thể biết trước dữ liệu tương lai để tính trung bình trước được. Do đó, tập Test trong dự án phải được đối xử hệt như khách hàng mới trong tương lai: Hoàn toàn đứng ngoài mọi phép tính trung bình/thống kê của tập Train.

In [ ]:
from sklearn.model_selection import train_test_split

# Chuẩn bị Features (X) và Target (y)
X = df_encoded[['Age', 'Salary', 'Education_Encoded', 'City_Hanoi', 'City_HCM']]
y = df_encoded['Purchased_Target']

# 1. Chia Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")

# 2. Scaler chuẩn quy trình
scaler = StandardScaler()

# Chỉ FIT và TRANSFORM trên Train Set
X_train_scaled = scaler.fit_transform(X_train)

# Chỉ TRANSFORM trên Test Set (KHÔNG gọi fit!)
X_test_scaled = scaler.transform(X_test)

print("\n--- X_train SAU KHI SCALED CHUẨN ---")
print(X_train_scaled[:3])

Kích thước X_train: (6, 5)
Kích thước X_test: (2, 5)

--- X_train SAU KHI SCALED CHUẨN ---
[[-1.03912236 -0.70925794 -0.52223297  1.         -0.70710678]
 [ 1.47488335  1.97442074  0.52223297 -1.          1.41421356]
 [-0.03352008 -0.1725222   1.5666989   1.         -0.70710678]]


: 